# Data Extraction
This notebook covers...



#### Install sec edgar package in your virtual environment if you haven't already done so. You can do this by running the following command in your terminal:
```pip install -U sec-edgar-downloader jupyter notebook```

In [14]:
import sys
sys.path.insert(0, str(ROOT))
from src.data_loader import extract

In [4]:
from pathlib import Path
ROOT = Path.cwd().parent                     # notebooks/ -> project root
print(Path.cwd()) #current folder
print(ROOT, (ROOT / "src").exists()) #root

target_folder = str(ROOT / "data" / "raw")                # downloader adds its own sec-edgar-filings folder
FILING_DIR    = ROOT / "data/raw/sec-edgar-filings/sec-edgar-filings"   # adjust to match what you see on disk
out_dir       = ROOT / "data/processed"

/Users/noirem/REPOS/MineTheGap/notebooks
/Users/noirem/REPOS/MineTheGap True


In [5]:
import os
from tqdm.notebook import tqdm  # Specialized notebook progress bar
from sec_edgar_downloader import Downloader

target_folder = os.path.expanduser(FILING_DIR)


# 12 companies to download 10-K filings for tickers AMZN, IBM, JPM, MCD, META, TSLA, TGT, & WMT (2021 2025 FY returns year before)
tickers = [
    #"TGT",   # Target
    "MSFT",  # Microsoft
    "PG",    # P&G
    #"TSLA",  # Tesla
    #"META",  # Meta
    #"WMT",   # Walmart
    #"AMZN",  # Amazon
    #"IBM",   # IBM
    "COST",  #Costco
    #MCD",   # McDonald's
    #"JPM",   # JPMorgan Chase
    "AAPL"   # Apple
]

# Initialize downloader with my identity (required by the SEC)
downloader = Downloader(
    "Noire Meyers", "noireefua@gmail.com",
     download_folder=target_folder,
)

# Download the 2020 10-k fillings for Target, Microsoft, P&G, Tesla, Meta, Walmart, Amazon, IBM, Costco, McDonald's, JPMorgan Chase, and Apple on a loop
print(f"Starting bulk download to: {target_folder}\n")

for ticker in tickers:
    print(f"Downloading 2020 10-K for: {ticker}...")
    try:
        downloader.get(
            "10-K", 
            ticker, 
            after="2020-01-01", 
            before="2020-12-31"
        )
    except Exception as e:
        print(f"Error downloading {ticker}: {e}")

print("\nAll downloads completed successfully!")

#same for 2024 10-k filings
print(f"Starting bulk download to: {target_folder}\n")

for ticker in tickers:
    print(f"Downloading 2024 10-K for: {ticker}...")
    try:
        downloader.get(
            "10-K", 
            ticker, 
            after="2024-01-01", 
            before="2024-12-31"
        )
    except Exception as e:
        print(f"Error downloading {ticker}: {e}")

print("\nAll downloads completed successfully!")

Starting bulk download to: /Users/noirem/REPOS/MineTheGap/data/raw/sec-edgar-filings/sec-edgar-filings


All downloads completed successfully!
Starting bulk download to: /Users/noirem/REPOS/MineTheGap/data/raw/sec-edgar-filings/sec-edgar-filings


All downloads completed successfully!


### The next step is to verify the data in the 10-K passages just downloaded
in terminal put the following

```
source .venv/bin/activate
pip install -r requirements.txt
pip install pytest
python -m spacy download en_core_web_sm   # only because tests/test_environment.py checks for it
```

Fiscal year is defined by the filing’s period-of-report date, with January and February year-ends assigned to the prior year. 

**Data Validation**
**AMZN, IBM, JPM, MCD, META, TSLA** end their FY on Dec 31st, and file in feburary of the next calendar year. 
This returns FY2019, FY2023 for a 2020 filing date in the SEC-EDGAR-API

**TGT and WMT** end their FY late janurary (1/30) or early febuarary. So a 10-K filed in 2020 really covers from i.e. 1/30/2019-1/30/2020.
This returns FY2019, FY2023 for a 2020 filing date in the SEC-EDGAR-API

My solution was to download their respective FY2021 & 2025FY

if loaded properly every values should be 1. a 0 means your file exists but not on the right path

In [6]:
from pathlib import Path    
import pandas as pd
from datetime import date     
from bs4 import BeautifulSoup #HTML Parser
import re #RegEx for extracting passages


def fiscal_year_end(period_end):
    """
    Returns selected accession FY(Fiscal Year)
    
    """
    return period_end.year - 1 if period_end.month <= 2 else  period_end.year #Edge case: Jan/Feb FY endings belong to previous year

def building_inventory(filings_dir):
    """
    Input: FILINGS_DIR()

    Output: table of company tickers by year & the count of each one. Run to verify data has loaded in correctly

    if loaded properly every values should be 1. a 0 means your file exists but not on the right path
    """
    rows = []

    for path in sorted(filings_dir.rglob("full-submission.txt")): 
        if "10-K" not in path.parts:
            continue  #If 10-k isn't found in the search sequence, continue
        ticker = path.parts[path.parts.index("10-K") - 1].upper()
        raw = path.read_text(encoding="utf-8", errors="ignore")
        period_date = re.search(r"CONFORMED PERIOD OF REPORT:\s*(\d{4})(\d{2})(\d{2})",raw) #date search for period of report

        if not period_date:
            print(f"Warning: No period data found for {ticker} \n path:f{path}")
            continue
        period = date(*map(int, period_date.groups()))
        rows.append(dict(
        ticker=ticker, 
        fiscal_year=fiscal_year_end(period),
        period=period,
        accession=path.parent.name,
        path=str(path),
        size_mb=round(path.stat().st_size/ 1e6,2)))

    return pd.DataFrame(rows, columns=["ticker", "fiscal_year", "period", "accession", "path", "size_mb"])

inv = building_inventory(FILING_DIR)
print(inv.groupby(["ticker","fiscal_year"]).size().unstack(fill_value=0))


fiscal_year  2020  2024
ticker                 
AAPL            1     1
AMZN            1     1
COST            1     1
IBM             1     1
JPM             1     1
MCD             1     1
META            1     1
MSFT            1     1
PG              1     1
TGT             1     1
TSLA            1     1
WMT             1     1


### Extraction Function
Scraping the HTML body so that we only get what we need, the 10-K. Then we filter it down to the DEI (or acknowledge the lack thereof) language

In [7]:
def ten_K(raw):
    for doc in re.findall(r"<DOCUMENT>(.*?)</DOCUMENT>", raw, flags=re.S):
        if re.search(r"<TYPE>\s*10-K\s*$", doc, flags=re.M):
            m = re.search(r"<TEXT>(.*?)(?:</TEXT>|$)",doc, flags=re.S)
            return m.group(1) if m else doc
    return None

### Candidate Passage CSV
This returns a dataframe in processed that counts how many DEI related passages were detected in each FY

In [13]:
BLOCKS = ["p", "div", "li", "td"]

def block_texts(soup):
    out=[]
    for el in soup.find_all(BLOCKS):
        if el.find(BLOCKS):
            continue
        t=re.sub(r"\s+", " ", el.get_text(" ", strip=True))
        t=re.sub(r"\s+([,.;:])", r"\1", t)
        if t:
            out.append(t)
    return out
import re
KEYWORDS = re.compile(r"\b(diversity|inclusion|inclusive|DEI|belonging|racial|African[- ]American|"
                      r"underrepresented|Black(?! Friday|[-\u2010-\u2013 ]Scholes))\b", re.IGNORECASE)

def section_of(text, current):
    s = text.lower()
    if re.match(r"item\s*1a\b", s): return "Item 1A"
    if re.match(r"item\s*(1b|2)\b", s): return "other"
    if re.match(r"item\s*1(?!\w)", s): return "Item 1"
    return current

def extract(raw, company, fy):
    html = ten_K(raw)
    if html is None:
        return []
    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script", "style"]):
        t.decompose()
    rows, seen, section = [], set(), "unknown"
    for text in block_texts(soup):
        section = section_of(text, section)
        if 80 <= len(text) <= 2000 and KEYWORDS.search(text) and text not in seen:
            seen.add(text)
            rows.append(dict(company=company, fiscal_year=fy, section=section,
                             text=text, label="", labeler=""))
    return rows

all_rows = []
for r in inv[inv.fiscal_year.isin([2020, 2024])].itertuples():
    raw = Path(r.path).read_text(encoding="utf-8", errors="ignore")
    all_rows += extract(raw, r.ticker, r.fiscal_year)
df = pd.DataFrame(all_rows, columns=["company", "fiscal_year", "section", "text", "label", "labeler"])
df.to_csv(out_dir / "passages_candidates.csv", index=False)

counts = df.groupby(["company", "fiscal_year"]).size().unstack(fill_value=0)
print(counts)


fiscal_year  2020  2024
company                
AAPL            0     6
AMZN            5     3
COST            1     1
JPM            16    15
MCD            10    10
META            4     4
MSFT            6    10
PG              4     7
TGT             2     5
TSLA            5     4
WMT             3     5


Validation testing: 

In [48]:
sample = df.sample(10, random_state=0)
for t in sample.text:
    print(t[:1000], "\n")

    #first sample: no, probably picked up "inclusive"
    #second: yes
    #3,4 no, picked up diversity and black
    #5 yes off of technicality. sustainiblity makes technology more accessible hence implictly DEI (in the middle)
    #6 i dont know but im leaning toward no
    #7 no
    #8 yes
    #9 idk
    #10 no
df.duplicated(["company", "fiscal_year", "text"]).sum()


(b) Movement in portfolio levels (inclusive of rule changes) refers to: for Credit risk RWA, changes in book size, changes in composition and credit quality, market movements, and deductions for excess eligible allowances for credit losses not eligible for inclusion in Tier 2 capital; for Market risk RWA, changes in position and market movements; and for Operational risk RWA, updates to cumulative losses and macroeconomic model inputs. 

Beginning in 2021, the Company is incorporating quantitative human capital management related metrics to annual incentive compensation for its executives. In addition to the Company’s financial performance, executives will be measured on their ability to champion our core values, improve diversity representation within leadership roles for both women and historically underrepresented groups, and create a strong culture of inclusion within the Company. 

Operating income was $14.5 billion and $22.9 billion for 2019 and 2020. We believe that operating in

np.int64(0)